# === Install dependencies (for Colab use) ===

In [1]:
!pip install -q tensorflow pandas scikit-learn

# === Imports ===

In [2]:
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from google.colab import files
print(f"pandas: {pd.__version__}, sklearn: {sklearn.__version__}, tensorflow: {tf.__version__}")

pandas: 2.2.2, sklearn: 1.6.1, tensorflow: 2.19.0


# === Upload files ===

In [3]:
uploaded = files.upload()
input_file = "data.csv"

Saving data.csv to data.csv


# === Load data ===

In [4]:
df = pd.read_csv(input_file)
print("Amount of rows: " + str(len(df)))
df.head()

Amount of rows: 86400


,throughput,retransmits,snd_cwnd,snd_wnd,rttvar,rtt
0,2.096516e+06,0.4,52707.2,104550.4,95311.8,411035.0
1,8.388736e+05,11.8,35331.2,241766.4,15066.2,439374.8
2,4.190399e+05,3.8,32724.8,304742.4,22191.2,391426.4
3,8.388224e+05,0.0,38227.2,408371.2,19608.2,492914.8
4,6.292677e+05,0.0,40833.6,408371.2,9764.4,483598.0


# === Create input and output dataframes ===

In [5]:
y = df['rtt']
X = df.drop(columns=['rtt'])
X.head()

,throughput,retransmits,snd_cwnd,snd_wnd,rttvar
0,2.096516e+06,0.4,52707.2,104550.4,95311.8
1,8.388736e+05,11.8,35331.2,241766.4,15066.2
2,4.190399e+05,3.8,32724.8,304742.4,22191.2
3,8.388224e+05,0.0,38227.2,408371.2,19608.2
4,6.292677e+05,0.0,40833.6,408371.2,9764.4


# === Kleinste und größte RTT-Werte ===

In [6]:
print(f"Smallest RTT: {y.min()}, largest RTT: {y.max()}")

Smallest RTT: 304972.2, largest RTT: 4780211.0


# === Normalize inputs ===

In [7]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
df_scaled = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
df_scaled.head()


,throughput,retransmits,snd_cwnd,snd_wnd,rttvar
0,2.574507,-0.459631,2.815023,-8.542893,3.211800
1,0.436265,3.500389,0.084571,-7.954768,-0.308763
2,-0.277536,0.721427,-0.324996,-7.684844,0.003828
3,0.436178,-0.598579,0.539647,-7.240678,-0.109495
4,0.079893,-0.598579,0.949214,-7.240678,-0.541365


# === Normalize outputs (Min-max normalization) ===

In [8]:
y_min = y.min()
y_max = y.max()
y_norm = (y - y_min) / (y_max - y_min)
y_norm.head()

,rtt
0,0.023700
1,0.030032
2,0.019318
3,0.041996
4,0.039914


# === Train/Validation/Test split ===

In [9]:
X_train_full, X_test, y_train_full, y_test_full = train_test_split(X_scaled, y_norm.values, test_size=0.3, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42) # 20% -> validation set (X_val, y_val)

# === Build the model ===

In [10]:
model = Sequential([Dense(256, activation='relu', input_shape=(X_train.shape[1],)),Dense(128, activation='relu'), Dense(64, activation='relu'), Dense(64, activation='relu'), Dense(32, activation='relu'), Dense(1)])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# === Callback for R² tracking ===

In [12]:
class R2Callback(tf.keras.callbacks.Callback):
  def __init__(self, X_val, y_val, y_min, y_max):
    super().__init__()
    self.X_val = X_val
    self.y_val = y_val
    self.y_min = y_min
    self.y_max = y_max

  def on_epoch_end(self, epoch, logs=None):
    y_pred = self.model.predict(self.X_val, verbose=0)
    # Denormalize
    y_pred_denorm = y_pred * (self.y_max - self.y_min) + self.y_min
    y_val_denorm = self.y_val * (self.y_max - self.y_min) + self.y_min
    r2 = r2_score(y_val_denorm, y_pred_denorm)
    print(f"Epoch {epoch + 1} - R² Score (Val): {r2:.4f}")

# === Train model ===

In [13]:
r2_callback = R2Callback(X_val, y_val, y_min, y_max)
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_data=(X_val, y_val), callbacks=[r2_callback], verbose=1)

Epoch 1/50
1499/1512 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.7110e-04 - mae: 0.0092Epoch 1 - R² Score (Val): 0.3546
1512/1512 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - loss: 2.7019e-04 - mae: 0.0092 - val_loss: 1.6728e-04 - val_mae: 0.0072
Epoch 2/50
1509/1512 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 1.6532e-04 - mae: 0.0074Epoch 2 - R² Score (Val): 0.5482
1512/1512 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - loss: 1.6524e-04 - mae: 0.0074 - val_loss: 1.1711e-04 - val_mae: 0.0066
Epoch 3/50
1497/1512 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.0044e-04 - mae: 0.0078Epoch 3 - R² Score (Val): 0.5315
1512/1512 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - loss: 1.9988e-04 - mae: 0.0078 - val_loss: 1.2142e-04 - val_mae: 0.0066
Epoch 4/50
1498/1512 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 1.1492e-04 - mae: 0.0069Epoch 4 - R² Score (Val): 0.4430
1512/1512 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - loss: 1.1493e-04 - mae: 0.0069 - val_loss: 1.4435e-04 - val_mae: 0.0071
Epoch 5/50
1506/1512 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - 

# === Final evaluation on test set ===

In [14]:
loss, mae = model.evaluate(X_test, y_test_full, verbose=1)
print(f"\nTest Loss (MSE): {loss:.4f}")
print(f"Test MAE: {mae:.4f}")

810/810 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.1027e-04 - mae: 0.0069

Test Loss (MSE): 0.0001
Test MAE: 0.0068


# === Final R² on test set ===

In [15]:
y_pred_test = model.predict(X_test)
y_pred_test_denorm = y_pred_test * (y_max - y_min) + y_min
y_test_denorm = y_test_full * (y_max - y_min) + y_min
r2_test = r2_score(y_test_denorm, y_pred_test_denorm)
print(f"Final R² Score (Test Set): {r2_test:.4f}")

810/810 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Final R² Score (Test Set): 0.5839
